
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 데모 - 검색을 위한 AI Search 구축

## 개요
환영합니다! 이 데모에서는 Databricks을 이용한 문서 검색용 AI Search 솔루션을 만드는 방법을 살펴보겠습니다. 문서 조각을 의미 벡터로 변환하는 방법, AI Search 인덱스 생성, 고급 검색 및 재순위 기법을 활용해 검색 정밀도를 높이는 과정을 살펴보겠습니다.

실제 검색 시나리오에서는 비구조화된 문서를 검색 가능한 의미 벡터로 변환하면 더 정확하고 맥락 인식에 맞는 결과가 나옵니다. 이 워크플로는 키워드가 정확히 일치하지 않아도 관련 정보를 효율적으로 찾을 수 있게 해줍니다.

## 학습 목표
- Foundation Model APIs의 GTE 모델을 사용하여 문서 임베딩을 compute하는 단계를 **식별**합니다.
- SDK 및 UI 메서드를 사용하여 AI Search 인덱스를 **구성**하고 생성합니다.
- 문서 경로별 필터링을 포함하여 쿼리, 하이브리드, 전체 텍스트 검색 방법을 **구현**하고 비교하세요.
- 재순위화를 통해 검색 정밀도를 **개선**하고 검색 품질에 미치는 영향을 이해합니다.
- 계산 비용, 정확도 및 인덱스 갱신 전략의 균형을 맞추기 위한 모범 사례를 **적용**합니다.

## 요구 사항
- 미리 생성된 **AI Search 엔드포인트**. 이것은 미리 생성되었습니다.
- **서버리스 Compute (환경 버전 5)**. [여기](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)에 따라 적절한 환경 버전을 선택하세요.
- 서버리스 compute 구성의 **의존성**에 필요한 라이브러리가 추가됩니다.
- 임베딩 생성을 위한 Foundation Model APIs 접근.
- AI Search 인덱스를 생성하고 관리할 수 있는 적절한 권한 부여.

## 준비

아래 코드를 실행하여 필요한 라이브러리를 설치하고 교실 환경을 구성하세요. 이 단계는 모든 의존성이 사용 가능하고 워크스페이스가 데모 준비가 완료되도록 보장합니다.


In [0]:
%run ../Includes/Classroom-Setup-03

## A. 소스 테이블 준비

AI Search은 소스 테이블에 change data feed(CDF)가 활성화되어 있어야 합니다. 테이블에 이미 이 기능이 활성화되어 있다면 변경할 필요가 없습니다. 그렇지 않다면 다음과 같이 활성화할 수 있습니다.

또한, AI Search 인덱스를 생성할 때 필요한 테이블의 고유 ID가 필요하다는 점도 주의해야 합니다.

In [0]:
# AI Search 동기화를 위해 change data feed 활성화
spark.sql(f"ALTER TABLE {docs_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# 표 구조를 이해하기 위해 샘플 데이터를 표시하세요
display(spark.sql(f"SELECT * FROM {docs_table} LIMIT 5"))

## B. 문서 청크를 위한 임베딩 컴퓨팅

임베딩은 의미적 의미를 포착하는 고차원 벡터 표현으로, 강력한 유사성 검색과 검색을 가능하게 합니다. Databricks에서는 임베딩을 통해 키워드가 정확히 일치하지 않더라도 맥락상 적합한 문서 청크를 찾을 수 있습니다.

Databricks는 임베딩 생성을 위한 두 가지 주요 접근법을 지원합니다:
* **관리형 임베딩:** AI Search는 임베딩을 자동으로 계산하고 관리하여 설정 및 유지보수를 간소화합니다. 대부분의 사용 사례에서 권장되는 접근법입니다.
* **수동 임베딩:** 외부에서 임베딩을 생성할 수 있습니다(MLflow 배포, Hugging Face, OpenAI, 등) 그리고 열에 저장할 수 있습니다. 대규모 데이터셋의 경우, 텍스트 열의 각 행마다 임베딩을 계산하기 위해 Spark UDF를 사용할 수 있습니다.

이번 데모에서는 **managed embeddings**를 사용하여 AI Search이 임베딩을 계산하고 유지할 수 있도록 합니다. 이 접근법은 워크플로를 간소화하고 Databricks AI Search 기능과의 호환성을 보장합니다.

In [0]:
import mlflow.deployments

# 임베딩 모델 접근을 위한 배포 클라이언트를 초기화하세요
deploy_client = mlflow.deployments.get_deploy_client("databricks")

# 샘플 질문에 임베딩 생성하기
question = "How Generative AI impacts humans?"
response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [question]})
embeddings = [e["embedding"] for e in response.data]

# 임베딩 정보 표시
print("Embedding for question:", embeddings[0])
print("Embedding shape:", len(embeddings[0]))

**💡 질문:** 임베딩 형태는 `1024` 무엇을 의미하나요? 우리가 사용하는 임베딩 모델에서는 이 부분을 변경할 수 있을까요?

## C. AI Search 인덱스 생성

임베딩이 생겼으니, 빠르고 정확한 검색을 가능하게 하는 AI Search 인덱스를 만들 예정입니다. Databricks는 두 가지 주요 접근법을 지원합니다:

- **SDK 방법:** databricks-vectorsearch SDK을 사용하여 계산된 임베딩이 포함된 인덱스를 생성합니다.
- **UI 메서드:** Databricks UI를 사용하여 관리되거나 계산된 임베딩이 포함된 인덱스를 생성합니다.

설정 섹션에 정의된 미리 생성된 AI Search 엔드포인트를 사용할 것입니다. 엔드포인트 생성 방법에 대한 자세한 내용은 [엔드포인트 문서](https://docs.databricks.com/aws/en/vector-search/create-vector-search#create-a-vector-search-endpoint-using-the-ui)를 참고하세요.


### C1. SDK를 통한 인덱스 생성

이 단계에서는 Databricks SDK를 사용하여 AI Search 인덱스를 만듭니다. 관리형 임베딩을 사용해 Databricks가 각 청크에 대해 자동으로 계산하고 벡터 표현을 유지할 수 있게 합니다.

자세한 내용은 [AI Search SDK 문서](https://api-docs.databricks.com/python/vector-search/index.html)를 참조하세요.

**참고:** 인덱스 갱신 모드는 업데이트 요구사항에 따라 수동 또는 동기화로 설정할 수 있습니다.

In [0]:
from databricks.vector_search.client import VectorSearchClient

# AI Search 클라이언트를 초기화하세요
vsc = VectorSearchClient(disable_notice=True)

# 3단계 명명 규칙을 사용하여 인덱스 이름을 정의합니다
index_name = f"{catalog}.{schema}.docs_chunked_index"

# Delta 동기화를 포함한 관리된 임베딩으로 인덱스를 생성합니다
vsc.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=index_name,
    source_table_name=docs_table,
    primary_key='id',
    embedding_source_column="chunk",
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",
)
print(f"Index '{index_name}' created for table '{docs_table}' using endpoint '{vector_search_endpoint}'.")

### C1. UI를 통한 인덱스 생성 (선택 사항)

또한 Databricks UI를 사용하여 AI Search 인덱스를 생성할 수 있으며, 이는 관리형 임베딩과 계산된 임베딩을 모두 지원합니다. 이 방법은 그래픽 인터페이스를 선호하거나 관리형 임베딩 Workflows를 활용하려는 사용자에게 권장됩니다.

단계별 설명은 [AI Search UI 문서](https://docs.databricks.com/aws/en/vector-search/create-vector-search#create-index-using-the-ui)을 참조하세요.

1. 왼쪽 사이드바에서 **카탈로그 (Catalog)** 를 클릭하여 Catalog Explorer를 엽니다.
2. Delta 테이블을 찾아서 선택하세요.
3. **생성 (Create)** (오른쪽 상단)를 클릭하고 **AI Search 인덱스 (AI Search index)** 를 선택하세요.
4. 대화 상자에서 다음을 구성하세요:
   * **이름:** 3단계 이름 (`<catalog>.<schema>.<name>`) 을 입력하세요. 드롭다운에서 카탈로그와 스키마를 선택하고 텍스트 상자에 이름을 입력하세요.
   * **인덱스 유형** `Hybrid`
   * **기본 키:** 고유 ID 열을 선택하세요.
   * **임베딩 소스:** **임베딩 컴퓨트 (Compute embeddings)** 을 선택하세요
     - **Embedding source column:** `chunk` 선택하세요
   * **계산된 임베딩:** 생성된 임베딩을 테이블에 저장하도록 토글 전환.
   * **AI Search 엔드포인트:** 엔드포인트를 선택하세요.
   * **동기화 모드:** *연속*(자동 동기화) 또는 *트리거*(수동 동기화)를 선택하세요. 스토리지 최적화 엔드포인트는 *트리거*만 지원합니다.
    * **고급 설정(Advanced settings):**
      - **Embedding model:** `databricks-gte-large-en` 선택하세요   
      - **색인할 열(Columns to index):** (표준 엔드포인트만 해당) 포함할 열을 선택하거나, 모든 열을 동기화하려면 비워 둡니다.
5. **생성 (Create)** 를 클릭하고 인덱스 생성 진행 상황을 모니터링하세요.

## D. 검색 방법: 쿼리, 하이브리드, 그리고 전체 텍스트 검색

인덱스가 설정되면, 이제 다양한 방법으로 검색을 수행할 수 있습니다:

- **쿼리 검색:** 임베딩을 사용해 의미적으로 유사한 청크를 찾습니다.
- **하이브리드 검색:** 의미 기반 검색과 키워드 기반 검색을 결합하여 관련성을 향상시킵니다.
- **전체 텍스트 검색:** 정확한 키워드 일치를 기반으로 청크를 검색합니다.

또한 특정 문서를 타겟팅하기 위해 필드별로 `path` 결과를 필터링하는 방법도 시연할 것입니다.

In [0]:
# 검색 수행을 위한 AI Search 인덱스를 얻으세요
index = vsc.get_index(index_name=index_name)
print(index.describe())

### D1. 쿼리 검색: 유사성 탐색

이 단계에서는 AI Search 인덱스를 사용해 의미 탐색을 수행할 것입니다. 이 방법은 정확히 일치하지 않는 키워드가 있어도 쿼리와 맥락상 유사한 문서 청크를 불러옵니다. 이 방법을 사용해 단순한 키워드가 아니라 의미에 기반한 관련 정보를 찾을 것입니다.

In [0]:
query_text = "How does the Orion system prevent overheating during continuous operation?"
results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3
)
display(results)

### D2. 하이브리드 검색: 의미 + 키워드

하이브리드 검색은 의미적 유사성과 키워드 매칭을 결합합니다. 이 접근법은 맥락적 관련성과 정확한 키워드 히트를 균형 있게 맞추고 싶을 때 유용하며, 검색 결과의 정밀도를 향상시킵니다.

In [0]:
query_text = "ISO 13849-1에 따른 안전 검증에 대해 설명해 주세요."
results_hybrid = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    query_type="hybrid",
    num_results=5
)
display(results_hybrid)

모델 임베딩은 의미론적으로는 안전성과 검증에 초점을 맞출 수 있지만, 임베딩이 주로 일반 영어 텍스트로 학습되었다면 구체적인 표준 참조를 놓칠 수 있습니다.

**"ISO 13849-1"** 의 키워드 필터링은 규정 준수 표준이 언급된 청크만 검색되도록 보장합니다. 참고: 첫 번째 결과에는 **"ISO 13849-1"** 가 포함되어 있지만 다른 결과에는 포함되지 않습니다.

### D3. 전체 텍스트 검색: 키워드만

전체 텍스트 검색은 정확한 키워드 일치만을 기반으로 문서 조각을 검색합니다. 검색어에 대해 매우 정확하고 문자 그대로의 결과가 필요할 때 이 방법을 사용하세요.

**🚨 중요:** 전체 텍스트 검색은 **현재 베타 버전**이며, 사용 전에 workspace에서 반드시 활성화되어야 합니다. 이 기능을 활성화한 후 아래 코드 셀을 **건너뛰기 취소**할 수 있습니다. **미리보기** 기능을 활성화하는 방법은 [이 문서 페이지](https://docs.databricks.com/aws/en/admin/workspace-settings/manage-previews#-manage-account-level-previews)에서 확인할 수 있습니다.

In [0]:
%skip
query_text = "PID coefficients"
results_fulltext = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    query_type="FULL_TEXT",
    num_results=5
)
display(results_fulltext)


### D4. 경로별 필터링: 특정 문서 대상으로 하기

필터링을 통해 특정 문서에 대한 결과를 반환할 수 있습니다. 예를 들어, 검색 결과를 `path` 필드별로 필터링하여 특정 문서만 대상으로 할 수 있습니다. 이는 데이터세트 내에서 특정 파일이나 문서로 검색을 제한하고 싶을 때 유용합니다.

여기서는 `LIKE` 필터를 사용할 거예요. 필터링이 문자열 내 여백으로 구분된 토큰과 일치한다는 것을 유의해 주세요. 다른 지원되는 필터는 [문서](https://docs.databricks.com/aws/en/vector-search/query-vector-search)에서 확인할 수 있습니다.

In [0]:
query_text = "How does the Orion system prevent overheating during continuous operation?"
filtered_results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    filters={"path LIKE" : "dbfs:/Volumes/main/default/documents/03_Orion_Motion_Controller_Firmware_Guide_v6.pdf"}, # TODO: 다른 카탈로그와 스키마를 사용했다면 경로를 변경하세요
    num_results=3
)

display(filtered_results)

**💡 추가 필터 예시:**

AI Search에는 다양한 필터 유형을 사용할 수 있습니다:

```Python
# 정확 경로 일치에 의한 필터링
filters={"path NOT": "specific/document/path.pdf"}

# 숫자 범위로 필터링하세요 (숫자 메타데이터가 있을 경우)
filters={"page_number >": 10, "page_number <": 50}
```

## E. 재순위를 통한 정확도 향상

임베딩은 의미적으로 유사한 내용을 찾는 데 강력하지만, 때로는 의미는 가깝지만 맥락에서는 약한 결과를 반환할 수 있습니다. 재순위 조정은 더 맥락 인식 있는 모델이나 추가 신호를 사용해 상위 결과를 재평가하여 정밀도를 높이는 데 도움을 줍니다.

**왜 재순위하는가?**
- 임베딩 기반 탐색은 의미론적으로 유사하지만 맥락상 무관한 청크를 드러낼 수 있습니다.
- 재순위 매기는 종종 교차인코더나 LLM을 사용하여 가장 관련성 높은 결과를 우선순위로 정하기 위해 2차 점수 산정 단계를 적용합니다.
- Databricks에서는 내장된 리랭커를 사용해 유사도 검색에서 상위 N개의 결과를 재평가하고 재정렬할 수 있으며, 이를 통해 더 깊은 맥락적 이해를 활용할 수 있습니다.
- 이는 특히 미묘한 쿼리와 중요한 사용 사례에서 최종 출력물의 품질을 향상시킵니다.

**트레이드오프:** 재순위 조정은 계산 비용을 증가시키지만 고가치 쿼리의 정확도를 크게 향상시킬 수 있습니다. 성능과 정확성의 균형을 맞추기 위해 리랭킹을 선택적으로 사용하세요.

In [0]:
# 예시: DatabricksReranker를 사용해 의미 검색으로 상위 N개 결과를 재랭크하기

from databricks.vector_search.reranker import DatabricksReranker

query_text = "How does the Orion system prevent overheating during continuous operation?"
results_reranked = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=5,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)

display(results_reranked)

**💡 질문:** 이 결과들을 이 섹션 시작 부분의 유사성 검색으로 반환한 결과와 비교해 보세요. 개선된 점이 보이나요?

## F. AI Search의 모범 사례

1. **가능하면 임베딩 차원을 최소화하세요**
   더 높은 차원 임베딩(예: 1024-1536)은 더 많은 뉘앙스를 포착할 수 있지만 지연을 증가시키고 처리량을 감소시킵니다. 검색 품질을 유지하는 가장 낮은 차원을 선택하세요.
   *예시:* 768-dim 모델과 384-dim 모델을 테스트해 유사한 검색 정확도를 찾는다면, 더 빠른 쿼리를 위해 384-dim 버전을 선호해야 합니다.

2. **쿼리당 중간 정도(예: 10-100)를 유지 `num_results` 하세요**
   너무 많은 결과를 요청하면 스캔과 지연이 증가합니다; 문서에서는 사용 사례가 이를 정당화하지 않는 한 이 범위 내에서 유지할 것을 권장합니다.
   *예시:* 기본적으로 `num_results=50`을 사용하고 `num_results=5000` 대신하세요.

3. **올바른 endpoint SKU를 선택하고 인덱스 크기를 적절히 설정하세요**
   벡터 수, 차원, 쿼리 지연, 비용을 기준으로 "표준" 엔드포인트와 "스토리지 최적화" 엔드포인트 중 선택할 수 있습니다. 또한 인덱스 크기가 AI Search 유닛의 능력 내에 유지되어 최적의 지연 시간을 확보하세요.
   *예시:* 768차원에서 200만 벡터 미만의 경우 표준 SKU를 사용하세요; 1,000만 벡터 초과의 경우, Storage-Optimized를 고려해 보세요.

4. **필터와 메타데이터를 사용해 검색 범위를 좁히세요**
   메타데이터(예: 문서 유형, 경로 접두사, 출처)를 첨부하면 검색을 `filters` 제한하고 관련 없는 청크를 회수하지 않아 관련성과 성능을 향상시킬 수 있습니다.
   *예시:* `"document_type":"manual"`로 필터링하여 “정비 주기” 쿼리 시 매뉴얼만 검색되도록 합니다.

5. **속도를 위해 ANN 검색을 선호합니다; 도메인 키워드가 중요할 때** 하이브리드(벡터 + 키워드)를 사용하세요
   근사 최근이웃(ANN) 검색은 가장 높은 QPS와 가장 낮은 지연 시간을 제공합니다; 하이브리드 검색은 키워드 관련성(예: 법률 표준 코드)이 매우 중요할 때만 사용해야 합니다.
   *예시:* 일반적인 "센서 재보정 방법" 쿼리에는 ANN을 사용하세요; 쿼리가 "ISO 13849-1"과 같은 특정 기준과 일치해야 할 때는 하이브리드를 사용하세요.

더 많은 모범 사례는 [Databricks AI Search 문서](https://docs.databricks.com/aws/en/generative-ai/vector-search-best-practices)를 참조하세요.

## G. 요약

이 데모에서는 Databricks에서 문서 검색을 위한 AI Search 솔루션을 구축하는 종단 간 과정을 탐구했습니다:

* Foundation Model APIs의 GTE 모델을 사용하여 문서 청크에 대한 의미 임베딩을 계산했습니다.
* SDK와 UI 메서드를 모두 사용하여 미리 생성된 엔드포인트를 활용해 AI Search 인덱스를 생성함.
* 문서 경로별 필터링을 포함한 쿼리, 하이브리드 및 전체 텍스트 검색 방법을 시연함.
* 재순위를 통해 검색 정밀도를 향상시키며, 임베딩이 의미론적으로 근접하지만 맥락상 약한 매칭을 반환할 때 그 가치를 강조합니다.
* 계산 비용, 정확도, 임베딩 차원성, 청킹 전략, 인덱스 갱신 모드의 균형을 맞추기 위한 모범 사례를 검토함.

이러한 단계와 모범 사례를 따르면 데이터와 비즈니스 요구에 맞춰 확장되는 견고하고 정밀한 검색 시스템을 구현할 수 있습니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>